In [1]:
import torch.nn as nn
import torch.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.preprocessing import StandardScaler
import pickle, copy, os, torch

In [ ]:
import torch
import torch.nn as nn

class LSTMblock(nn.Module):
    def __init__(self, input_size=14, hidden_size=128):
        super().__init__()

        self.hidden_size = hidden_size
        self.W = nn.Parameter(torch.randn(4 * hidden_size, input_size))
        self.U = nn.Parameter(torch.randn(4 * hidden_size, hidden_size))
        self.b = nn.Parameter(torch.zeros(4 * hidden_size))

        nn.init.xavier_uniform_(self.W)
        nn.init.orthogonal_(self.U)

        self.sigm = nn.Sigmoid()
        self.tanh = nn.Tanh()

    def forward(self, inputs):
        B, T, _ = inputs.shape
        h = torch.zeros(B, self.hidden_size, device=inputs.device)
        c = torch.zeros(B, self.hidden_size, device=inputs.device)

        outputs = []
        for t in range(T):  
            x_t = inputs[:, t, :]  
            gates = (x_t @ self.W.T) + (h @ self.U.T) + self.b  
            f, i, g, o = torch.chunk(gates, 4, dim=1)

            f = self.sigm(f)
            i = self.sigm(i)
            g = self.tanh(g)
            o = self.sigm(o)

            c = f * c + i * g
            h = o * self.tanh(c)
            outputs.append(h.unsqueeze(1))

        return torch.cat(outputs, dim=1), (c, h)  


class VanillaLSTM(nn.Module):
    def __init__(self, input_size=14, hidden_size=128):
        super().__init__()
        self.LSTM = LSTMblock(input_size, hidden_size)
        self.Head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 144)
        )

    def forward(self, inputs):  
        seq_out, (c, h) = self.LSTM(inputs)  
        output = self.Head(h) 
        return output


In [3]:
class ClimateLog(Dataset):
    def __init__(self, root, split="train"):
        super().__init__()
        self.root = root
        self.scaler = StandardScaler()

        df = pd.read_csv(self.root).copy()
        del df['Date Time']

        df = df.iloc[:-4320].copy()

        n = len(df)
        train_end = int(0.8 * n)
        val_end   = int(0.9 * n)

        train_df = df.iloc[:train_end].copy()
        val_df   = df.iloc[train_end:val_end].copy()
        test_df  = df.iloc[val_end:].copy()

        self.train_dataset = torch.tensor(
            self.scaler.fit_transform(train_df.values),
            dtype=torch.float32
        )
        self.val_dataset = torch.tensor(
            self.scaler.transform(val_df.values),
            dtype=torch.float32
        )
        self.test_dataset = torch.tensor(
            self.scaler.transform(test_df.values),
            dtype=torch.float32
        )

        self.stride = 24
        self.split = split 

    def __len__(self):
        dataset = self._get_dataset()
        return (len(dataset) - 4320 - 144 + 1) // self.stride

    def __getitem__(self, index):
        index *= self.stride
        dataset = self._get_dataset()

        inputs = dataset[index:index+4320, :]
        labels = dataset[index+4320:index+4320+144, 2]

        return inputs, labels

    def _get_dataset(self):
        if self.split == "train":
            return self.train_dataset
        elif self.split == "val":
            return self.val_dataset
        elif self.split == "test":
            return self.test_dataset
        else:
            raise ValueError(f"Unknown split: {self.split}")


In [4]:
BATCH_SIZE = 128
DEVICE = torch.device('cuda')

In [5]:
train_dataset = ClimateLog("/kaggle/input/datasets/quphine/jena-climate/jena_climate_2009_2016.csv", split='train')
val_dataset = ClimateLog("/kaggle/input/datasets/quphine/jena-climate/jena_climate_2009_2016.csv", split='val')

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

In [6]:
def train_model(
    model, optimizer, criterion, train_loader, val_loader,
    scheduler, loss_dict, acc_dict, n_epochs, device,
    checkpoint_path="latest_checkpoint.pth"
):
    start_epoch = 0
    best_loss    = 1000.0

    if torch.cuda.device_count() > 1 and not isinstance(model, nn.DataParallel):
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    model = model.to(device)

    if os.path.exists(checkpoint_path):
        print("Checkpoint found. Resuming training...")
        ckpt = torch.load(checkpoint_path, map_location=device)
        (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        best_loss    = ckpt["best_loss"]
        loss_dict.update(ckpt["loss_dict"])
        acc_dict.update(ckpt["acc_dict"])
        print(f"Resumed from Epoch {start_epoch}")

    best_wts = copy.deepcopy(
        (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
    )

    for epoch in range(start_epoch, n_epochs):
        print(f"\nEpoch [{epoch+1}/{n_epochs}]  lr={scheduler.get_last_lr()[0]:.6f}")

        model.train()
        train_loss = 0

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss    += loss.item() * images.size(0)

            if (i + 1) % 50 == 0 or (i + 1) == len(train_loader):
                print(f"  Step [{i+1}/{len(train_loader)}] Loss: {loss.item():.4f}")

        epoch_train_loss = train_loss    / len(train_loader.dataset)
        loss_dict["train"].append(epoch_train_loss)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs    = model(images)
                loss       = criterion(outputs, labels)
                val_loss  += loss.item() * images.size(0)

        epoch_val_loss = val_loss    / len(val_loader.dataset)
        loss_dict["val"].append(epoch_val_loss)

        scheduler.step()

        if epoch_val_loss < best_loss:
            best_loss = epoch_val_loss
            best_wts = copy.deepcopy(
                (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
            )

        print("-" * 60)
        print(f"Train Loss: {epoch_train_loss:.4f} ")
        print(f"Val   Loss: {epoch_val_loss:.4f} (best: {best_loss:.4f})")
        print("-" * 60)

        raw_state = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        torch.save({
            "epoch":                epoch,
            "model_state_dict":     raw_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_loss":             best_loss,
            "loss_dict":            loss_dict,
            "acc_dict":             acc_dict,
        }, checkpoint_path)

    torch.save(best_wts, "model_wts.pth")
    with open("acc_loss.bin", "wb") as f:
        pickle.dump([loss_dict, acc_dict], f)

    (model.module if isinstance(model, nn.DataParallel) else model).load_state_dict(best_wts)
    print(f"\nTraining Complete! Best Val loss: {best_loss:.4f}")
    return model

In [ ]:
device = 'cuda'
num_epochs = 50

criterion = nn.MSELoss()
model = VanillaLSTM()
model.to(device)



optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-1)

'''
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)
'''
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,  
    eta_min=1e-6,        
)
KAGGLE_INPUT_CHECKPOINT = "/kaggle/input/models/quphine/v2-cifar/transformers/default/1/latest_checkpoint.pth"
OUTPUT_CHECKPOINT_PATH  = "/kaggle/working/latest_checkpoint.pth"

import shutil
if os.path.exists(KAGGLE_INPUT_CHECKPOINT) and not os.path.exists(OUTPUT_CHECKPOINT_PATH):
    print("Linking Kaggle input checkpoint...")
    shutil.copy(KAGGLE_INPUT_CHECKPOINT, OUTPUT_CHECKPOINT_PATH)

loss_dict = {'train': [], 'val': []}
acc_dict  = {'train': [], 'val': []}

train_model(
    model=model,
    device=device,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    scheduler=scheduler,
    loss_dict=loss_dict,
    acc_dict=acc_dict,
    n_epochs=num_epochs,
    checkpoint_path=OUTPUT_CHECKPOINT_PATH,
)

Using 2 GPUs

Epoch [1/50]  lr=0.000200
  Step [50/107] Loss: 0.3947
  Step [100/107] Loss: 0.2405
  Step [107/107] Loss: 0.2673
------------------------------------------------------------
Train Loss: 0.5159 
Val   Loss: 0.2439 (best: 0.2439)
------------------------------------------------------------

Epoch [2/50]  lr=0.000200
  Step [50/107] Loss: 0.2317
  Step [100/107] Loss: 0.1996
  Step [107/107] Loss: 0.2250
------------------------------------------------------------
Train Loss: 0.2177 
Val   Loss: 0.2269 (best: 0.2269)
------------------------------------------------------------

Epoch [3/50]  lr=0.000199
  Step [50/107] Loss: 0.2050
  Step [100/107] Loss: 0.2211
  Step [107/107] Loss: 0.2172
------------------------------------------------------------
Train Loss: 0.2009 
Val   Loss: 0.2213 (best: 0.2213)
------------------------------------------------------------

Epoch [4/50]  lr=0.000198
  Step [50/107] Loss: 0.1781
  Step [100/107] Loss: 0.1717
  Step [107/107] Loss: 0.

DataParallel(
  (module): VanillaLSTM(
    (LSTM): LSTMblock(
      (sigm): Sigmoid()
      (tanh): Tanh()
    )
    (Head): Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU(inplace=True)
      (2): Linear(in_features=256, out_features=144, bias=True)
    )
  )
)